
# Mini Project 1 – Radio Signal Propagation Using Sionna RT

## Objective

In this project, we use NVIDIA Sionna RT to investigate radio signal propagation in free-space and urban environments.

We explore four main topics:

1. **Free-Space Propagation:** How distance affects received signal strength (RSS) and path gain.
2. **Building Materials:** How concrete and wood affect signal propagation.
3. **LOS vs NLOS:** How building obstructions affect RSS at similar transmitter-receiver distances in the Munich scene.
4. **Propagation Mechanisms:** Which propagation mechanism provides the strongest received signal at a selected NLOS receiver position.

We use ray-tracing simulations, numerical calculations, and visualizations to compare the results and understand how different factors affect wireless signal propagation.


## Experiment 1 – Free-Space Propagation

### Research Questions

1. If the Tx–Rx distance doubles in ideal free space, does received power become half?
2. How do RSS and path gain change with distance?

### Setup

We used one transmitter and three receivers at different distances: 30 m, 60 m, and 120 m.

We used a frequency of 3.5 GHz and a transmit power of 30 dBm.

We measured RSS and path gain for each receiver.

In [ ]:
import sionna.rt
from sionna.rt import (
    load_scene,
    Transmitter,
    Receiver,
    PlanarArray,
    PathSolver
)

In [ ]:
# 1. Load scene
scene = load_scene()

In [ ]:
# 2. Set frequency
scene.frequency = 3.5e9

In [ ]:
# 3. Configure TX antenna
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V"
)

In [ ]:
# 4. Configure RX antenna
scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V"
)

In [ ]:
# 5. Add Base station
BS = Transmitter(
    name="BS1",
    position=[8.5,21,27]
)
scene.add(BS)

In [ ]:
# 6. Add receiver
RX1 = Receiver(
    name="RX1",
    position=[45, 90, 1.5]
)
scene.add(RX1)


In [ ]:
# 7. Add receiver
RX2 = Receiver(
    name="RX2",
    position=[85.9, 10, 1.5]
)
scene.add(RX2)


In [ ]:
# RX3 - Twice the 3D distance of RX1
RX3 = Receiver(
    name="RX3",
    position=[84.36513669, 164.41628581, 1.5]
)
scene.add(RX3)

In [ ]:
# 8. Show TX, RX 
scene.preview()

In [ ]:
import numpy as np
#Store the positions as NumPy arrays
#We use them to calculate the exact 3D distance

# Get the positions directly from Sionna
BS_position = BS.position.numpy()
RX1_position = RX1.position.numpy()
RX2_position = RX2.position.numpy()
RX3_position = RX3.position.numpy()

# Calculate the Euclidean distance
d1 = np.linalg.norm(RX1_position - BS_position)
d2 = np.linalg.norm(RX2_position - BS_position)
d3 = np.linalg.norm(RX3_position - BS_position)

print("Distance BS1 → RX1:", d1, "m")
print("Distance BS1 → RX2:", d2, "m")
print("Distance BS1 → RX3:", d3, "m")

In [ ]:
import numpy as np
c = 299792458.0
frequency = 3.5e9
wavelength = c / frequency
tx_power_dbm = 30.0

# Calculate free-space path loss 
fspl1_db = 20 * np.log10(
    4 * np.pi * d1/ wavelength
)
fspl2_db = 20 * np.log10(
    4 * np.pi * d2 / wavelength
)
fspl3_db = 20 * np.log10(
    4 * np.pi * d3 / wavelength
)
#fspl results
print(f"RX1 fspl1_db: {fspl1_db:.2f} dB")
print(f"RX2 fspl2_db: {fspl2_db:.2f} dB")
print(f"RX3 fspl3_db: {fspl3_db:.2f} dB")


In [ ]:
# Received powers

rss1_dbm = tx_power_dbm - fspl1_db
rss2_dbm = tx_power_dbm - fspl2_db
rss3_dbm = tx_power_dbm - fspl3_db

print(f"RX1 RSS: {rss1_dbm:.2f} dBm")
print(f"RX2 RSS: {rss2_dbm:.2f} dBm")
print(f"RX3 RSS: {rss3_dbm:.2f} dBm")


In [ ]:
# Path Gain results(Path gain is the negative of path loss.)
path_gain1_db = rss1_dbm - tx_power_dbm
path_gain2_db = rss2_dbm - tx_power_dbm
path_gain3_db = rss3_dbm - tx_power_dbm


print(f"Path Gain 1: {path_gain1_db:.2f}, dB")
print(f"Path Gain 2: {path_gain2_db:.2f}, dB")
print(f"Path Gain 3: {path_gain3_db:.2f}, dB")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Distances and calculated RSS values
distances = [d1, d2, d3]
rss_values = [rss1_dbm, rss2_dbm, rss3_dbm]
receivers = ["RX1", "RX2", "RX3"]
colors = ["royalblue", "darkorange", "green"]

plt.figure(figsize=(9, 5))

# Plot each receiver separately
for i in range(3):
    plt.scatter(
        distances[i],
        rss_values[i],
        color=colors[i],
        s=90,
        label=receivers[i],
        zorder=3
    )

    plt.annotate(
        f"{receivers[i]}\n{distances[i]:.2f} m\n{rss_values[i]:.2f} dBm",
        (distances[i], rss_values[i]),
        xytext=(10, 15 if i != 1 else -45),
        textcoords="offset points",
        fontsize=9,
        color=colors[i]
    )

# Connect the points in increasing distance order
order = np.argsort(distances)

plt.plot(
    np.array(distances)[order],
    np.array(rss_values)[order],
    linestyle="--",
    color="gray",
    alpha=0.6
)

plt.title("Received Signal Strength vs. Distance (Free Space)")
plt.xlabel("3D Tx-Rx Distance (m)")
plt.ylabel("RSS (dBm)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### Results

We calculated RSS and path gain for three receivers in a free-space environment using a transmit power of 30 dBm.



| Receiver | Distance (m) | Path Gain (dB) | RSS (dBm) |
|:---:|---:|---:|---:|
| RX1 | 82.12 | -81.62 | -51.62 |
| RX2 | 82.23 | -81.63 | -51.63 |
| RX3 | 164.24 | -87.64 | -57.64 |



### Conclusion

In free-space propagation, received signal strength (RSS) and path gain decrease as the distance between the transmitter and receiver increases.

RX1 and RX2 have almost identical RSS values because their distances from the transmitter are nearly equal.

When the distance doubles from 82.12 m to 164.24 m, RSS decreases by approximately 6 dB. This means the received power becomes approximately one-fourth of its original value, not half.

The results are consistent with the theoretical free-space path loss model.


## Experiment 2 – Building Materials

### Research Question

How does changing the building material affect received signal strength (RSS) and path gain?

### Setup

We use the same transmitter and RX3 positions as in Experiment 1.

We place a building between the transmitter and RX3 and compare two materials: concrete and wood.

We keep the frequency, antenna settings, building geometry, and device positions constant.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mitsuba as mi

from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    SceneObject,
    RadioMaterial,
    PathSolver
)

In [ ]:
def create_material_scene():

    # Create empty scene
    s = load_scene()
    s.frequency = 3.5e9

    # Configure antennas
    s.tx_array = PlanarArray(
        num_rows=1,
        num_cols=1,
        pattern="iso",
        polarization="V"
    )

    s.rx_array = PlanarArray(
        num_rows=1,
        num_cols=1,
        pattern="iso",
        polarization="V"
    )

    # Transmitter position
    s.add(Transmitter(
        name="tx",
        position=[8.5, 21, 27]
    ))

    # Receiver position
    s.add(Receiver(
        name="rx3",
        position=[84.36513669, 164.41628581, 1.5]
    ))

    return s

In [ ]:
# Define concrete material
concrete = RadioMaterial(
    name="my_concrete",
    relative_permittivity=5.31,
    conductivity=0.0326,
    color=(0.28, 0.47, 0.77)  # Blue
)

In [ ]:
# Create concrete scene
scene_concrete = create_material_scene()

# Building geometry
building_center = [46.43, 92.71, 15]
building_scale = [5, 5, 15]

# Create concrete building
building_concrete = SceneObject(
    mi_mesh=mi.load_dict({
        "type": "cube",
        "to_world": mi.ScalarTransform4f()
            .translate(building_center)
            .scale(building_scale)
    }),
    name="building",
    radio_material=concrete
)

scene_concrete.add(building_concrete)

print("Concrete building added successfully!")

In [ ]:
# Define wood material
wood = RadioMaterial(
    name="my_wood",
    relative_permittivity=1.99,
    conductivity=0.0047,
    color=(0.65, 0.37, 0.22)  # Brown
)

In [ ]:
# Create wood scene
scene_wood = create_material_scene()

# Create wood building
building_wood = SceneObject(
    mi_mesh=mi.load_dict({
        "type": "cube",
        "to_world": mi.ScalarTransform4f()
            .translate(building_center)
            .scale(building_scale)
    }),
    name="building",
    radio_material=wood
)

scene_wood.add(building_wood)

print("Wood building added successfully!")

In [ ]:
solver = PathSolver()

In [ ]:
def calculate_signal(scene, tx_power_dbm=30):

    # Calculate propagation paths
    paths = solver(
        scene=scene,
        max_depth=3,
        los=True,
        specular_reflection=True,
        diffuse_reflection=False,
        refraction=True,
        diffraction=False
    )

    # Get channel coefficients and delays
    a, tau = paths.cir(
        out_type="numpy",
        normalize_delays=False
    )

    coefficients = a[0, 0, 0, 0, :, 0]
    valid = tau[0, 0, :] >= 0

    # Calculate path gain
    path_gain_linear = np.sum(
        np.abs(coefficients[valid])**2
    )

    if path_gain_linear > 0:
        path_gain_db = 10 * np.log10(path_gain_linear)
        rss_dbm = tx_power_dbm + path_gain_db
    else:
        path_gain_db = None
        rss_dbm = None

    return path_gain_db, rss_dbm, int(np.sum(valid)), paths

In [ ]:
# Calculate concrete results
path_gain_concrete, rss_concrete, paths_concrete_count, paths_concrete = calculate_signal(
    scene_concrete
)

# Calculate wood results
path_gain_wood, rss_wood, paths_wood_count, paths_wood = calculate_signal(
    scene_wood
)

# Display results
print("CONCRETE")
print("Path Gain:", path_gain_concrete, "dB")
print("RSS:", rss_concrete, "dBm")
print("Number of paths:", paths_concrete_count)

print("\nWOOD")
print("Path Gain:", path_gain_wood, "dB")
print("RSS:", rss_wood, "dBm")
print("Number of paths:", paths_wood_count)

In [ ]:
scene_concrete.preview(
    paths=paths_concrete,
    show_devices=True
)

In [ ]:
scene_wood.preview(
    paths=paths_wood,
    show_devices=True
)

In [ ]:
# Materials and RSS results
materials = ["Concrete", "Wood"]
rss_values = [rss_concrete, rss_wood]

if any(v is None for v in rss_values):
    print("No valid propagation path found for one or more materials.")

else:
    plt.figure(figsize=(7, 5))

    bars = plt.bar(
        materials,
        rss_values,
        color=["steelblue", "darkorange"],
        width=0.5
    )

    # Add RSS labels
    for bar, rss in zip(bars, rss_values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            rss + 2,
            f"{rss:.2f} dBm",
            ha="center",
            fontsize=11
        )

    plt.title("RSS Comparison: Concrete vs Wood")
    plt.ylabel("Received Signal Strength (dBm)")
    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


### Conclusion

The results show that building materials can affect received signal strength (RSS).

We compared concrete and wood using the same transmitter, receiver, building geometry, and frequency (3.5 GHz). Only the building material was changed.

The difference in RSS shows how the electromagnetic properties of a material can influence signal propagation.

This experiment demonstrates that received signal strength depends not only on distance but also on the materials and obstacles between the transmitter and receiver.



## Experiment 3 – Urban Propagation (LOS vs NLOS)

### Research Question

How does building obstruction affect received signal strength (RSS) when two receivers are at similar distances from the transmitter?

### Objective

We compare LOS and NLOS propagation using the Munich scene in Sionna RT.

We use one transmitter and two receivers:

- **RX1 (LOS):** The direct signal path is not blocked.
- **RX2 (NLOS):** A building blocks the direct signal path.

Both receivers are approximately 82 meters from the transmitter.

We keep the frequency, transmit power, and antenna settings constant to investigate the effects of building obstruction.


Similar Distances

Research Question

Why can two locations at similar distances from the transmitter have different received signal strengths?
Setup

We used the Munich scene with one transmitter and two nearby receivers.
RX2 and RX4 are only 3 meters apart. We kept the transmitter position and antenna settings the same.
We compared their distances, RSS, and path gain.

In [ ]:

# Load the Munich urban environment
scene_munich = load_scene(
    sionna.rt.scene.munich
)

# Set frequency
scene_munich.frequency = 3.5e9

print("Munich scene loaded successfully!")


In [ ]:

# Configure TX antenna
scene_munich.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V"
)

# Configure RX antenna
scene_munich.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V"
)


In [ ]:

# Create transmitter
tx_munich = Transmitter(
    name="tx",
    position=[8.5, 21, 27]
)
scene_munich.add(tx_munich)

# Create receiver 1 (LOS candidate)
rx1_munich = Receiver(
    name="rx1",
    position=[45, 90, 1.5]
)
scene_munich.add(rx1_munich)

# Create receiver 2 (NLOS candidate)
rx2_munich = Receiver(
    name="rx2",
    position=[85.9, 10, 1.5]
)
scene_munich.add(rx2_munich)

print("TX, RX1 and RX2 added successfully!")


In [ ]:

# Calculate 3D distances
tx_pos = np.array([8.5, 21, 27])
rx1_pos = np.array([45, 90, 1.5])
rx2_pos = np.array([85.9, 10, 1.5])

d_rx1 = np.linalg.norm(rx1_pos - tx_pos)
d_rx2 = np.linalg.norm(rx2_pos - tx_pos)

print(f"TX - RX1: {d_rx1:.2f} m")
print(f"TX - RX2: {d_rx2:.2f} m")
print(f"Distance difference: {abs(d_rx1 - d_rx2):.2f} m")


In [ ]:

# Calculate only direct propagation paths
direct_paths = solver(
    scene=scene_munich,
    max_depth=0,
    los=True
)

# Get channel coefficients
a_direct, _ = direct_paths.cir(out_type="numpy")

# Check LOS/NLOS conditions
for i, name in enumerate(scene_munich.receivers.keys()):

    # Check whether a direct path exists
    has_los = np.any(np.abs(a_direct[i]) > 0)

    status = "LOS" if has_los else "NLOS"

    print(f"{name}: {status}")


In [ ]:

# Calculate urban propagation paths
paths_munich = solver(
    scene=scene_munich,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=True,
    refraction=True,
    diffraction=True
)

print("Munich propagation paths calculated!")


In [ ]:

# Show calculated propagation paths
scene_munich.preview(
    paths=paths_munich,
    show_devices=True
)


In [ ]:

# Get channel coefficients
a_munich, tau_munich = paths_munich.cir(
    out_type="numpy"
)

tx_power_dbm = 30
rss_results = []
path_gain_results = []

# Calculate results for both receivers
for i, name in enumerate(scene_munich.receivers.keys()):

    coefficients = a_munich[i, 0, 0, 0, :, 0]
    valid = tau_munich[i, 0, :] >= 0

    path_gain_linear = np.sum(
        np.abs(coefficients[valid])**2
    )

    if path_gain_linear > 0:
        path_gain_db = 10 * np.log10(path_gain_linear)
        rss_dbm = tx_power_dbm + path_gain_db
    else:
        path_gain_db = np.nan
        rss_dbm = np.nan

    path_gain_results.append(path_gain_db)
    rss_results.append(rss_dbm)

    print(f"{name}: Path Gain = {path_gain_db:.2f} dB")
    print(f"{name}: RSS = {rss_dbm:.2f} dBm")


In [ ]:

# Receiver labels
receivers = ["RX1 (LOS)", "RX2 (NLOS)"]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    receivers,
    rss_results,
    color=["seagreen", "coral"],
    width=0.5
)

# Add RSS labels
for bar, rss in zip(bars, rss_results):
    if np.isfinite(rss):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            rss + 2,
            f"{rss:.2f} dBm",
            ha="center",
            fontsize=11
        )

plt.title("RSS Comparison: LOS vs NLOS")
plt.ylabel("Received Signal Strength (dBm)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()



### Conclusion

This experiment demonstrates how building obstructions affect received signal strength (RSS) in an urban environment.

RX1 (LOS) and RX2 (NLOS) are located at approximately the same distance from the transmitter (82 m), but their propagation conditions are different.

In the LOS condition, the signal can travel directly from the transmitter to the receiver. In the NLOS condition, buildings block the direct path, so the signal must rely on other propagation mechanisms, such as reflection and diffraction.

The experiment shows that distance alone does not determine received signal strength. Building obstructions and propagation conditions also play an important role.



## Experiment 4 – Propagation Mechanisms Comparison (NLOS)

### Research Question

**Which propagation mechanism contributes the most to received signal strength (RSS) at our selected NLOS receiver position in the Munich scene?**

### Objective

The objective of this experiment is to compare the contributions of different radio propagation mechanisms to the received signal strength (RSS) at a specific NLOS location.

### Setup

We use the same Munich scene and transmitter-receiver configuration as in Experiment 3.

- **Transmitter (TX):** [8.5, 21, 27]
- **Receiver (RX2 – NLOS):** [85.9, 10, 1.5]
- **Frequency:** 3.5 GHz
- **Transmit Power:** 30 dBm

We test the following propagation mechanisms separately:

1. Line of Sight (LOS)
2. Specular Reflection
3. Diffuse Reflection (Scattering)
4. Diffraction
5. Refraction / Transmission

Finally, we enable all mechanisms and compare the results.

### Important Note

The results are specific to the selected TX and RX2 positions in the Munich scene.

The dominant propagation mechanism may change depending on the receiver location, building geometry, material properties, and simulation settings.

Therefore, the results should not be generalized to all NLOS environments.


In [ ]:

# Propagation mechanisms to compare
mechanisms = {
    "LOS Only": {"los": True},
    "Specular Reflection": {"specular_reflection": True},
    "Diffuse Reflection": {"diffuse_reflection": True},
    "Diffraction": {"diffraction": True},
    "Refraction": {"refraction": True}
}

print("Propagation mechanisms defined successfully!")


In [ ]:

def test_mechanism(settings):

    # Disable all propagation mechanisms
    config = {
        "los": False,
        "specular_reflection": False,
        "diffuse_reflection": False,
        "diffraction": False,
        "refraction": False
    }

    # Enable selected mechanism
    config.update(settings)

    # Calculate propagation paths
    paths = solver(
        scene=scene_munich,
        max_depth=3,
        seed=42,
        **config
    )

    # Get channel coefficients
    a, tau = paths.cir(out_type="numpy")

    # RX2 is the second receiver (index 1)
    coefficients = a[1, 0, 0, 0, :, 0]
    valid = tau[1, 0, :] >= 0

    # Calculate path gain
    power = np.sum(
        np.abs(coefficients[valid])**2
    )

    # Calculate RSS
    if power > 0:
        rss = 30 + 10 * np.log10(power)
    else:
        rss = np.nan

    return rss, int(np.sum(valid))


In [ ]:

# Store simulation results
mechanism_results = {}

# Test each mechanism separately
for name, settings in mechanisms.items():

    rss, count = test_mechanism(settings)

    mechanism_results[name] = {
        "RSS": rss,
        "Paths": count
    }

    if np.isfinite(rss):
        print(f"{name}: RSS = {rss:.2f} dBm, Paths = {count}")
    else:
        print(f"{name}: No path found")


In [ ]:
# Enable all propagation mechanisms
all_settings = {
    "los": True,
    "specular_reflection": True,
    "diffuse_reflection": True,
    "diffraction": True,
    "refraction": True
}

# Calculate RSS with all mechanisms enabled
rss_all, paths_all = test_mechanism(all_settings)

# Store the result
mechanism_results["All Mechanisms"] = {
    "RSS": rss_all,
    "Paths": paths_all
}

if np.isfinite(rss_all):
    print(f"All Mechanisms: RSS = {rss_all:.2f} dBm")
    print(f"Number of paths: {paths_all}")
else:
    print("All Mechanisms: No path found")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

names = list(mechanism_results.keys())
rss_values = [mechanism_results[name]["RSS"] for name in names]

colors = [
    "gray", "steelblue", "gray",
    "mediumpurple", "darkorange", "seagreen"
]

x = np.arange(len(names))

fig, ax = plt.subplots(figsize=(12, 5.5))

for i, rss in enumerate(rss_values):

    if np.isfinite(rss):
        ax.bar(
            i, rss,
            color=colors[i],
            width=0.55
        )

        # Position label just below the bar endpoint
        ax.text(
            i, rss - 2,
            f"{rss:.2f} dBm",
            ha="center",
            va="top",
            fontsize=10,
            fontweight="bold"
        )

    else:
        ax.text(
            i, -10,
            "No path",
            ha="center",
            va="center",
            fontsize=10,
            color="dimgray"
        )

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha="right")

ax.set_title(
    "Propagation Mechanisms – RSS at NLOS Receiver (RX2)",
    fontsize=13
)
ax.set_ylabel("Received Signal Strength (dBm)")
ax.set_ylim(-100, 5)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()



### Conclusion

The results show that different propagation mechanisms contribute differently to received signal strength (RSS) at our selected NLOS receiver position in the Munich scene.

- **Refraction / Transmission** produced the strongest signal among the individual mechanisms (-55.98 dBm).
- **Specular Reflection** produced -66.72 dBm, while **Diffraction** produced a weaker signal (-80.69 dBm).
- **LOS and Diffuse Reflection** did not produce valid propagation paths in these simulations.
- When all mechanisms were enabled, the highest overall RSS (-55.04 dBm) was obtained with 48 propagation paths.

An important observation is that a higher number of propagation paths does not necessarily mean stronger received power. For example, refraction produced only one path but a higher RSS than diffraction, which produced nine paths.

**Overall, refraction was the strongest individual mechanism in this specific simulation. However, the dominant mechanism may change depending on the receiver location, building geometry, material properties, and simulation settings.**
